In [24]:
import math
import numpy as np
import pandas as pd

raw_df = pd.read_csv('PSCompPars_2026.03.31_11.13.09.csv')

C:\Users\shank\AppData\Local\Temp\ipykernel_15828\1519676046.py:5: DtypeWarning: Columns (82,87,92,104,109,114,119,124,129,150,155) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('PSCompPars_2026.03.31_11.13.09.csv')


In [25]:
raw_df.head()

,rowid,pl_name,hostname,pl_letter,hd_name,hip_name,tic_id,gaia_dr2_id,gaia_dr3_id,sy_snum,...,sy_kepmagerr1,sy_kepmagerr2,sy_kepmag_reflink,pl_nnotes,st_nphot,st_nrvc,st_nspec,pl_nespec,pl_ntranspec,pl_ndispec
0,1,11 Com b,11 Com,b,HD 107383,HIP 60202,TIC 72437047,Gaia DR2 3946945413106333696,Gaia DR3 3946945413106333696,2,...,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,2,1,2,0,0,0,0
1,2,11 UMi b,11 UMi,b,HD 136726,HIP 74793,TIC 230061010,Gaia DR2 1696798367260229376,Gaia DR3 1696798367260229376,1,...,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,0,1,1,0,0,0,0
2,3,14 And b,14 And,b,HD 221345,HIP 116076,TIC 333225860,Gaia DR2 1920113512486282240,Gaia DR3 1920113512486282240,1,...,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,0,1,1,0,0,0,0
3,4,14 Her b,14 Her,b,HD 145675,HIP 79248,TIC 219483057,Gaia DR2 1385293808145621504,Gaia DR3 1385293808145621504,1,...,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,1,1,4,1,0,0,0
4,5,16 Cyg B b,16 Cyg B,b,HD 186427,HIP 96901,TIC 27533327,Gaia DR2 2135550755683407232,Gaia DR3 2135550755683407232,3,...,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,5,1,4,3,0,0,0


In [26]:
def radius_mass_relation(pl_rade, pl_bmasse):
    if np.isnan(pl_bmasse) and not np.isnan(pl_rade):
        pl_bmasse = 0.9718 * (pl_rade ** 3.58)
    elif np.isnan(pl_rade) and not np.isnan(pl_bmasse):
        pl_rade = (pl_bmasse / 0.9718) ** (1 / 3.58)
    return pl_rade, pl_bmasse

radius_mass_rel = np.vectorize(radius_mass_relation)
result = radius_mass_rel(raw_df['pl_rade'],raw_df['pl_bmasse'])
raw_df['pl_rade'] = result[0]
raw_df['pl_bmasse'] = result[1]

raw_df = raw_df.dropna(subset=['pl_rade','pl_bmasse'], thresh=1)


In [27]:
def calculate_density(pl_bmasse, pl_rade):
    
    # Convert to SI units
    e_mass = 5.972e24   # kg
    e_rad = 6.371e6     # meters

    mass_kg   = pl_bmasse * e_mass
    radius_m  = pl_rade * e_rad

    # Volume in m3
    volume_m3 = (4/3) * np.pi * (radius_m ** 3)

    # Density in kg/m3 then convert to g/cm3
    density_kgm3 = mass_kg / volume_m3
    density_gcm3 = density_kgm3 / 1000  # kg/m3 → g/cm3

    return density_gcm3


density = np.vectorize(calculate_density)
raw_df['pl_dens'] = density(raw_df['pl_bmasse'], raw_df['pl_rade'])

# Filter out unrealistic densities (Earth is 5.51, densest known planets ~15-20)
raw_df = raw_df[raw_df['pl_dens'] <= 20]


In [28]:
def escape_velocity(pl_rade,pl_bmasse):
    e_mass = 5.972e24 #Earth Mass
    e_rad = 6.371e6   #Earth Radius
    G = 6.674e-11     #Universal Gravitatinal Constant

    pl_bmasse = pl_bmasse*e_mass
    pl_rade = pl_rade*e_rad

    esc_vel = np.sqrt((2 * 6.674e-11 * (pl_bmasse)) / (pl_rade))

    return esc_vel


esc_vel = np.vectorize(escape_velocity)
raw_df['pl_escvel'] = esc_vel(raw_df['pl_rade'], raw_df['pl_bmasse'])
raw_df.head()

,rowid,pl_name,hostname,pl_letter,hd_name,hip_name,tic_id,gaia_dr2_id,gaia_dr3_id,sy_snum,...,sy_kepmagerr2,sy_kepmag_reflink,pl_nnotes,st_nphot,st_nrvc,st_nspec,pl_nespec,pl_ntranspec,pl_ndispec,pl_escvel
0,1,11 Com b,11 Com,b,HD 107383,HIP 60202,TIC 72437047,Gaia DR2 3946945413106333696,Gaia DR3 3946945413106333696,2,...,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,2,1,2,0,0,0,0,224513.032911
1,2,11 UMi b,11 UMi,b,HD 136726,HIP 74793,TIC 230061010,Gaia DR2 1696798367260229376,Gaia DR3 1696798367260229376,1,...,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,0,1,1,0,0,0,0,218302.055996
2,3,14 And b,14 And,b,HD 221345,HIP 116076,TIC 333225860,Gaia DR2 1920113512486282240,Gaia DR3 1920113512486282240,1,...,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,0,1,1,0,0,0,0,103941.480329
3,4,14 Her b,14 Her,b,HD 145675,HIP 79248,TIC 219483057,Gaia DR2 1385293808145621504,Gaia DR3 1385293808145621504,1,...,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,1,1,4,1,0,0,0,168267.618575
4,5,16 Cyg B b,16 Cyg B,b,HD 186427,HIP 96901,TIC 27533327,Gaia DR2 2135550755683407232,Gaia DR3 2135550755683407232,3,...,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,5,1,4,3,0,0,0,72411.035294


In [29]:

def fill_derived_columns(row):
    """
    Derives missing st_lum, pl_orbsmax, and pl_orbper from available columns.
    """
    # st_lum
    if not np.isnan(row["st_lum"]):
        st_lum = row["st_lum"]
    elif not np.isnan(row["st_rad"]) and not np.isnan(row["st_teff"]):
        st_lum = 2 * np.log10(row["st_rad"]) + 4 * np.log10(row["st_teff"] / 5778)
    else:
        st_lum = np.nan

    # pl_orbsmax
    if not np.isnan(row["pl_orbsmax"]):
        pl_orbsmax = row["pl_orbsmax"]
    elif not np.isnan(row["pl_orbper"]) and not np.isnan(row["st_mass"]):
        pl_orbsmax = (row["st_mass"] * (row["pl_orbper"] / 365.25) ** 2) ** (1/3)
    else:
        pl_orbsmax = np.nan

    # pl_orbper
    if not np.isnan(row["pl_orbper"]):
        pl_orbper = row["pl_orbper"]
    elif not np.isnan(pl_orbsmax) and not np.isnan(row["st_mass"]):
        pl_orbper = ((pl_orbsmax ** 3) / row["st_mass"]) ** 0.5 * 365.25
    else:
        pl_orbper = np.nan

    return pd.Series({"st_lum": st_lum, "pl_orbsmax": pl_orbsmax, "pl_orbper": pl_orbper})

raw_df[["st_lum", "pl_orbsmax", "pl_orbper"]] = raw_df.apply(fill_derived_columns, axis=1)
raw_df = raw_df.dropna(subset=["st_lum", "pl_orbsmax"]).reset_index(drop=True)
print(f"Remaining after fill + drop: {len(raw_df)}")


Remaining after fill + drop: 5653


In [30]:
def calculate_hzd(pl_orbsmax, st_lum, pl_orbper, st_mass):
    if np.isnan(st_lum):
        return np.nan
    
    lum = 10 ** st_lum
    
    if np.isnan(pl_orbsmax):
        if not np.isnan(pl_orbper) and not np.isnan(st_mass):
            pl_orbsmax = ((pl_orbper / 365.25) ** 2 * st_mass) ** (1/3)
        else:
            return np.nan
    
    d_inner = 0.95 * np.sqrt(lum)
    d_outer = 1.67 * np.sqrt(lum)
    d_center = (d_inner + d_outer) / 2
    d_width  = (d_outer - d_inner) / 2
    hzd = (pl_orbsmax - d_center) / d_width
    
    return hzd

hzd_vec = np.vectorize(calculate_hzd)
raw_df['hzd'] = hzd_vec(raw_df['pl_orbsmax'], raw_df['st_lum'], raw_df['pl_orbper'], raw_df['st_mass'])


In [31]:
raw_df[['pl_name','hzd']]

,pl_name,hzd
0,11 Com b,-3.303362
1,11 UMi b,-3.379690
2,14 And b,-3.380044
3,14 Her b,5.763277
4,16 Cyg B b,0.483610
...,...,...
5648,ups And b,-3.549048
5649,ups And c,-2.383136
5650,ups And d,0.173833
5651,ups Leo b,-3.226255


In [32]:
raw_df = raw_df[raw_df['hzd'].between(-1, 1)].sort_values('hzd',ascending=True)

In [33]:
def calculate_sephi(pl_rade, pl_bmasse, pl_dens, pl_escvel, pl_eqt, pl_orbsmax, st_lum, st_teff, st_age, hzd):
    # --- L1: Telluric likelihood (using Zeng-Sasselov mass-radius grid) ---
    # Check if planet composition is consistent with rocky/telluric planets
    # Using simplified Zeng-Sasselov relation for Earth-like composition
    # Expected radius for given mass (Earth-like composition)
    if pl_bmasse <= 1.0:
        expected_radius = pl_bmasse ** 0.274  # Sub-Earth scaling
    elif pl_bmasse <= 4.0:
        expected_radius = pl_bmasse ** 0.306  # Super-Earth scaling
    else:
        expected_radius = pl_bmasse ** 0.274  # Fallback
    
    # Gaussian centered on expected radius for rocky composition
    radius_deviation = abs(pl_rade - expected_radius) / expected_radius
    L1 = np.exp(-(radius_deviation ** 2) / (2 * 0.25 ** 2))
    
    
    # --- L2: Atmosphere retention likelihood (asymmetric escape velocity) ---
    # Asymmetric Gaussian: stricter for low escape velocity (atmosphere loss)
    # Earth escape velocity = 11186 m/s
    escvel_ratio = pl_escvel / 11186.0
    
    if escvel_ratio < 1.0:
        # Below Earth: stricter penalty (atmosphere loss risk)
        sigma_low = 0.2
        L2 = np.exp(-((escvel_ratio - 1.0) ** 2) / (2 * sigma_low ** 2))
    else:
        # Above Earth: more lenient (atmosphere retention better)
        sigma_high = 2.553
        L2 = np.exp(-((escvel_ratio - 1.0) ** 2) / (2 * sigma_high ** 2))
    
    
    # --- L3: Liquid water likelihood (using Kopparapu HZ boundaries) ---
    # Calculate HZ boundaries from stellar temperature (Kopparapu et al. 2013)
    if not np.isnan(st_lum):
        lum_linear = 10 ** st_lum
    else:
        return np.nan
    
    # Kopparapu conservative HZ boundaries (empirical fits)
    # These are more accurate than simple sqrt(L) scaling
    if not np.isnan(st_teff):
        T_star = st_teff
        T_sun = 5780.0
        T_s = T_star - T_sun
        
        # Recent Venus (inner edge) - runaway greenhouse
        S_eff_inner = 1.7763 - 1.4316e-4 * T_s + 3.3954e-9 * T_s**2 - 3.2188e-12 * T_s**3
        d_inner = np.sqrt(lum_linear / S_eff_inner)
        
        # Early Mars (outer edge) - maximum greenhouse
        S_eff_outer = 0.3179 + 5.4513e-5 * T_s - 1.5028e-9 * T_s**2 - 1.0915e-12 * T_s**3
        d_outer = np.sqrt(lum_linear / S_eff_outer)
    else:
        # Fallback to simple scaling if T_eff unavailable
        d_inner = 0.95 * np.sqrt(lum_linear)
        d_outer = 1.67 * np.sqrt(lum_linear)
    
    # HZ center and width
    d_center = (d_inner + d_outer) / 2
    d_width = (d_outer - d_inner) / 2
    
    # Normalized distance from HZ center
    if not np.isnan(pl_orbsmax):
        hzd_raw = (pl_orbsmax - d_center) / d_width
        # Gaussian peaked at HZ center
        L3_orbital = np.exp(-(hzd_raw ** 2) / (2 * 0.5 ** 2))
    else:
        L3_orbital = 0.0
    
    # Temperature component (liquid water range: 273-373 K)
    T_optimal = 288.0  # Earth's mean surface temperature
    if pl_eqt >= 273 and pl_eqt <= 373:
        L3_temp = np.exp(-((pl_eqt - T_optimal) ** 2) / (2 * 50 ** 2))
    elif pl_eqt < 273:
        # Stricter penalty for frozen conditions
        L3_temp = np.exp(-((pl_eqt - 273) ** 2) / (2 * 30 ** 2))
    else:
        # Penalty for too hot
        L3_temp = np.exp(-((pl_eqt - 373) ** 2) / (2 * 50 ** 2))
    
    L3 = L3_orbital
    
    
    # --- L4: Magnetic field likelihood ---
    # Estimate tidal locking timescale
    # Simplified tidal locking criterion: P_rot / P_tidal_lock
    # For tidally locked planets, use Sano et al. scaling
    # For non-locked, use standard SOC (Scaling, Olson, Christensen) approximation
    
    # Estimate if planet is likely tidally locked
    # Rough criterion: close-in planets around low-mass stars
    if not np.isnan(st_lum):
        lum_linear = 10 ** st_lum
        # Tidal locking more likely for close orbits and low-mass stars
        tidal_lock_threshold = 0.1 * np.sqrt(lum_linear)  # Rough estimate
        
        if pl_orbsmax < tidal_lock_threshold:
            # Tidally locked: use Sano et al. (2016) scaling
            # Magnetic field depends on tidal heating
            # B ∝ (tidal heating flux)^(1/3)
            # Simplified: weaker fields for tidally locked planets
            mass_factor = (pl_bmasse / 1.0) ** 0.2  # Weak mass dependence
            L4 = 0.3 * mass_factor  # Reduced magnetic field likelihood
        else:
            # Not tidally locked: use SOC scaling
            # B ∝ (rotation rate)^(1/2) * (mass)^(1/3)
            # Assume rotation rate scales with mass for unknown rotation
            mass_factor = (pl_bmasse / 1.0) ** (1/3)
            
            # Age factor: younger stars have stronger fields
            if not np.isnan(st_age):
                # Magnetic field decays with age
                age_factor = np.exp(-((st_age - 4.5) ** 2) / (2 * 3.0 ** 2))
            else:
                age_factor = 0.5  # Neutral if age unknown
            
            L4 = mass_factor * age_factor
            L4 = min(L4, 1.0)  # Cap at 1.0
    else:
        # Fallback if luminosity unavailable
        L4 = 0.5
    
    
    # --- SEPHI: geometric mean of four sub-indexes ---
    sephi = (L1 * L2 * L3 * L4) ** (1/4)
    
    return sephi

sephi_vec = np.vectorize(calculate_sephi, otypes=[float])
raw_df['sephi'] = sephi_vec(
    raw_df['pl_rade'], raw_df['pl_bmasse'], raw_df['pl_dens'],
    raw_df['pl_escvel'], raw_df['pl_eqt'], raw_df['pl_orbsmax'],
    raw_df['st_lum'], raw_df['st_teff'], raw_df['st_age'], raw_df['hzd']
)


C:\Users\shank\AppData\Local\Temp\ipykernel_15828\872835256.py:49: RuntimeWarning: invalid value encountered in sqrt
  d_inner = np.sqrt(lum_linear / S_eff_inner)
C:\Users\shank\AppData\Local\Temp\ipykernel_15828\872835256.py:53: RuntimeWarning: invalid value encountered in sqrt
  d_outer = np.sqrt(lum_linear / S_eff_outer)


In [34]:
raw_df[['pl_name','hzd','sephi']].sort_values('sephi',ascending=False)


,pl_name,hzd,sephi
4584,LHS 1140 b,0.623947,0.974990
302,GJ 667 C f,0.062137,0.967638
3852,Kepler-442 b,-0.316727,0.962354
3527,Kepler-296 f,0.384182,0.927820
201,GJ 1061 d,-0.000859,0.927434
...,...,...,...
72,BD-08 2823 c,0.244326,0.000003
906,HD 210193 b,0.234744,0.000001
1283,HD 99109 b,0.118711,0.000000
820,HD 181720 b,0.049776,0.000000
